# 02 — Preparación y pipelines de cinco modelos
Este notebook carga la asignación congelada del EDA y construye pipelines **sin entrenarlos**. La lógica reutilizable está en `src/pipeline.py` y `src/splitting.py`.

| Modelo | Imputación | Escalado | Transformación baseline |
|---|---|---|---|
| Ridge | Mediana | StandardScaler | Ninguna |
| Random Forest | Mediana | Ninguno | Ninguna |
| XGBoost | Mediana | Ninguno | Ninguna |
| MLP | Mediana | StandardScaler | Ninguna |
| SVR | Mediana | StandardScaler | Ninguna |

Usamos imputación común aunque los árboles admitan faltantes en estas versiones. Es una decisión de baseline, no una exigencia de todos los estimadores. La mediana y el escalador se aprenden dentro de cada fold. Si una columna queda totalmente vacía en un fold, `keep_empty_features=True` conserva su posición y usa 0; no inventa una mediana.


In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.pipeline import construir_pipeline
from src.splitting import folds_por_grupos
config = json.loads((ROOT / "configs/pipelines.json").read_text())
snapshot = Path(json.loads((ROOT / "configs/dataset.local.json").read_text())["snapshot"]).name
carpeta = ROOT / "data/modelado" / snapshot
manifest = json.loads((carpeta / "manifest.json").read_text())
split_manifest = json.loads((carpeta / "split/manifest.json").read_text())
for archivo, esperado in [(carpeta / "datos_modelado.parquet", split_manifest["source_sha256"]),
                          (carpeta / "split/asignacion.parquet", split_manifest["assignment_sha256"])]:
    assert hashlib.sha256(archivo.read_bytes()).hexdigest() == esperado, f"Archivo modificado: {archivo.name}"
datos = pd.read_parquet(carpeta / "datos_modelado.parquet")
asignacion = pd.read_parquet(carpeta / "split/asignacion.parquet")
assert datos.index.equals(asignacion.index)
features = manifest["predictors"]
train = datos.loc[asignacion["PARTICION"].eq("train")].copy()
X_train = train[features]
y_train = train[manifest["target"]]
grupos_train = train[manifest["group"]]
solo_train = asignacion.loc[train.index, "ENTRENAMIENTO_OBLIGATORIO"]
print("Entrenamiento:", X_train.shape, "| Test reservado, sin analizar")


Entrenamiento: (3670, 173) | Test reservado, sin analizar


## 1. Validación por grupos
Cinco folds dentro de entrenamiento. Los 293 casos obligatorios se agregan a cada fold de entrenamiento y nunca a validación. Su similitud estructural sigue sin comprobarse. El MLP no usa `early_stopping` interno porque esa división aleatoria no respetaría nuestros grupos.


In [2]:
folds = folds_por_grupos(grupos_train, solo_train, n_splits=config["cv_folds"])
resumen_folds = pd.DataFrame([
    {"fold": i + 1, "train": len(tr), "validacion": len(va),
     "grupos_compartidos": len(set(grupos_train.iloc[tr]) & set(grupos_train.iloc[va])),
     "obligatorios_en_validacion": int(solo_train.iloc[va].sum())}
    for i, (tr, va) in enumerate(folds)
])
display(resumen_folds)


,fold,train,validacion,grupos_compartidos,obligatorios_en_validacion
0,1,2994,676,0,0
1,2,2994,676,0,0
2,3,2995,675,0,0
3,4,2995,675,0,0
4,5,2995,675,0,0


## 2. Pipelines baseline sin ajustar
Cada estimador recibe un objeto independiente. `ColumnTransformer` selecciona por nombre solo los predictores y descarta columnas adicionales. PKD y grupo no entran al modelo. Los hiperparámetros son los valores por defecto de las bibliotecas instaladas; solo se fija semilla y paralelismo.


In [3]:
pipelines = {nombre: construir_pipeline(nombre, features, seed=config["seed"], n_jobs=-1)
             for nombre in config["modelos"]}
for nombre, pipeline in pipelines.items():
    print(nombre)
    display(pipeline)


Ridge


Pipeline(steps=[('preparacion',
                 ColumnTransformer(transformers=[('originales',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median'))]),
                                                  ['N_LIGAND', 'N_RECEPTOR',
                                                   'MIN_DISTANCE_ANGSTROM',
                                                   'Q1_LIGAND_ANGSTROM',
                                                   'Q1_RECEPTOR_ANGSTROM',
                                                   'N_SELECTED_Q1_LIGAND',
                                                   'N_SELECTED_Q1_RECEPTOR',
                                                   'SELECTED_PAIRS_LT10',
                                                   'SELECTED_PAIR_COUN...
                                                   'LIGAND_CHEM_N_AA_E',
                                                   'LIGAND_CHEM_N_AA_F',
                                                   'LIGAND_CHEM_N_AA_G',
                                                   'LIGAND_CHEM_N_AA_H',
                                                   'LIGAND_CHEM_N_AA_I',
                                                   'LIGAND_CHEM_N_AA_K',
                                                   'LIGAND_CHEM_N_AA_L',
                                                   'LIGAND_CHEM_N_AA_M',
                                                   'LIGAND_CHEM_N_AA_N',
                                                   'LIGAND_CHEM_N_AA_P',
                                                   'LIGAND_CHEM_N_AA_Q',
                                                   'LIGAND_CHEM_N_AA_R',
                                                   'LIGAND_CHEM_N_AA_S', ...])],
                                   verbose_feature_names_out=False)),
                ('escalado', StandardScaler()), ('modelo', Ridge())])

RandomForestRegressor


Pipeline(steps=[('preparacion',
                 ColumnTransformer(transformers=[('originales',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median'))]),
                                                  ['N_LIGAND', 'N_RECEPTOR',
                                                   'MIN_DISTANCE_ANGSTROM',
                                                   'Q1_LIGAND_ANGSTROM',
                                                   'Q1_RECEPTOR_ANGSTROM',
                                                   'N_SELECTED_Q1_LIGAND',
                                                   'N_SELECTED_Q1_RECEPTOR',
                                                   'SELECTED_PAIRS_LT10',
                                                   'SELECTED_PAIR_COUN...
                                                   'LIGAND_CHEM_N_AA_G',
                                                   'LIGAND_CHEM_N_AA_H',
                                                   'LIGAND_CHEM_N_AA_I',
                                                   'LIGAND_CHEM_N_AA_K',
                                                   'LIGAND_CHEM_N_AA_L',
                                                   'LIGAND_CHEM_N_AA_M',
                                                   'LIGAND_CHEM_N_AA_N',
                                                   'LIGAND_CHEM_N_AA_P',
                                                   'LIGAND_CHEM_N_AA_Q',
                                                   'LIGAND_CHEM_N_AA_R',
                                                   'LIGAND_CHEM_N_AA_S', ...])],
                                   verbose_feature_names_out=False)),
                ('escalado', 'passthrough'),
                ('modelo', RandomForestRegressor(n_jobs=-1, random_state=42))])

XGBRegressor


Pipeline(steps=[('preparacion',
                 ColumnTransformer(transformers=[('originales',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median'))]),
                                                  ['N_LIGAND', 'N_RECEPTOR',
                                                   'MIN_DISTANCE_ANGSTROM',
                                                   'Q1_LIGAND_ANGSTROM',
                                                   'Q1_RECEPTOR_ANGSTROM',
                                                   'N_SELECTED_Q1_LIGAND',
                                                   'N_SELECTED_Q1_RECEPTOR',
                                                   'SELECTED_PAIRS_LT10',
                                                   'SELECTED_PAIR_COUN...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=None, n_jobs=-1,
                              num_parallel_tree=None, random_state=42, ...))])

MLPRegressor


Pipeline(steps=[('preparacion',
                 ColumnTransformer(transformers=[('originales',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median'))]),
                                                  ['N_LIGAND', 'N_RECEPTOR',
                                                   'MIN_DISTANCE_ANGSTROM',
                                                   'Q1_LIGAND_ANGSTROM',
                                                   'Q1_RECEPTOR_ANGSTROM',
                                                   'N_SELECTED_Q1_LIGAND',
                                                   'N_SELECTED_Q1_RECEPTOR',
                                                   'SELECTED_PAIRS_LT10',
                                                   'SELECTED_PAIR_COUN...
                                                   'LIGAND_CHEM_N_AA_F',
                                                   'LIGAND_CHEM_N_AA_G',
                                                   'LIGAND_CHEM_N_AA_H',
                                                   'LIGAND_CHEM_N_AA_I',
                                                   'LIGAND_CHEM_N_AA_K',
                                                   'LIGAND_CHEM_N_AA_L',
                                                   'LIGAND_CHEM_N_AA_M',
                                                   'LIGAND_CHEM_N_AA_N',
                                                   'LIGAND_CHEM_N_AA_P',
                                                   'LIGAND_CHEM_N_AA_Q',
                                                   'LIGAND_CHEM_N_AA_R',
                                                   'LIGAND_CHEM_N_AA_S', ...])],
                                   verbose_feature_names_out=False)),
                ('escalado', StandardScaler()),
                ('modelo', MLPRegressor(random_state=42))])

SVR


Pipeline(steps=[('preparacion',
                 ColumnTransformer(transformers=[('originales',
                                                  Pipeline(steps=[('imputacion',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median'))]),
                                                  ['N_LIGAND', 'N_RECEPTOR',
                                                   'MIN_DISTANCE_ANGSTROM',
                                                   'Q1_LIGAND_ANGSTROM',
                                                   'Q1_RECEPTOR_ANGSTROM',
                                                   'N_SELECTED_Q1_LIGAND',
                                                   'N_SELECTED_Q1_RECEPTOR',
                                                   'SELECTED_PAIRS_LT10',
                                                   'SELECTED_PAIR_COUN...
                                                   'LIGAND_CHEM_N_AA_E',
                                                   'LIGAND_CHEM_N_AA_F',
                                                   'LIGAND_CHEM_N_AA_G',
                                                   'LIGAND_CHEM_N_AA_H',
                                                   'LIGAND_CHEM_N_AA_I',
                                                   'LIGAND_CHEM_N_AA_K',
                                                   'LIGAND_CHEM_N_AA_L',
                                                   'LIGAND_CHEM_N_AA_M',
                                                   'LIGAND_CHEM_N_AA_N',
                                                   'LIGAND_CHEM_N_AA_P',
                                                   'LIGAND_CHEM_N_AA_Q',
                                                   'LIGAND_CHEM_N_AA_R',
                                                   'LIGAND_CHEM_N_AA_S', ...])],
                                   verbose_feature_names_out=False)),
                ('escalado', StandardScaler()), ('modelo', SVR())])

## 3. Transformaciones opcionales para comparar después
`construir_pipeline` acepta `transformacion="log1p"` o `"yeo-johnson"` y una lista explícita `columnas_transformar`. No las activamos automáticamente por estar entre las 15 más asimétricas.

- **log1p:** solo para variables no negativas por definición; rechaza negativos en entrenamiento e inferencia. No usar para senos, cosenos ni variables que puedan ser negativas. No resta mínimos ni recorta valores.
- **Yeo-Johnson:** admite valores negativos; sus parámetros se ajustan en cada fold. Revisar columnas constantes antes de usarlo.
- Los árboles quedan sin escalado; los otros tres modelos escalan después de la transformación opcional.

En `03_baseline_candidatos.ipynb` entrenaremos y compararemos los cinco pipelines usando estos folds y MLflow. Al usar paralelismo entre folds, configurar `n_jobs=1` en los estimadores para evitar competir por todos los cores.

Para deployment se guardará el **pipeline ajustado completo**; FastAPI cargará ese objeto y llamará a `predict` con las mismas columnas de entrada. Este notebook no entrena ni exporta un modelo ajustado. Las dos features derivadas se diseñan y evalúan en la sección 4 de este cuaderno.


## 4. Dos features derivadas y prueba de aporte (solo CV de entrenamiento)
1. `LOG1P_SELECTED_PAIR_COUNT = log(1 + SELECTED_PAIR_COUNT)`: motivada por la asimetría 10,79 del conteo en el EDA. Comprime la cola y permite al modelo lineal representar una relación con rendimientos decrecientes.
2. `PARTNER_SIZE_BALANCE = 2 × min(N_LIGAND, N_RECEPTOR)/(N_LIGAND + N_RECEPTOR)`: mide equilibrio de tamaños entre participantes (1: iguales; cerca de 0: muy desiguales). Es invariante a intercambiar participantes y aporta una relación no lineal de los tamaños originales.

Ambas se calculan por fila desde la estructura disponible antes del ensayo. No usan PKD ni estadísticas globales; son numéricas y no necesitan codificación categórica. Los faltantes siguen la imputación del pipeline. Una transformación no garantiza mejora: comparamos Ridge sin ellas, con cada una y con ambas, manteniendo los mismos folds. Esta prueba explora aporte y no sustituye retrospectivamente los 173 predictores usados en los cinco candidatos y tuning.


In [4]:
from src.pipeline import agregar_features_derivadas
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from threadpoolctl import threadpool_limits
from src.tracking import configurar_tracking
import mlflow
configurar_tracking()
resultados_derivadas = []
for incluir in [(), ("LOG1P_SELECTED_PAIR_COUNT",), ("PARTNER_SIZE_BALANCE",), ("LOG1P_SELECTED_PAIR_COUNT", "PARTNER_SIZE_BALANCE")]:
    nombre = " + ".join(incluir) or "sin_derivadas"
    candidato = Pipeline([
        ("derivadas", FunctionTransformer(agregar_features_derivadas, kw_args={"incluir":incluir})),
        ("estimador", construir_pipeline("Ridge", features+list(incluir), seed=config["seed"], n_jobs=1))
    ])
    with threadpool_limits(limits=1, user_api="blas"):
        scores = cross_validate(candidato, X_train, y_train, cv=folds,
                                scoring="neg_root_mean_squared_error",return_train_score=True,n_jobs=1,error_score="raise")
    fila = {"features":nombre,"rmse_cv":float((-scores["test_score"]).mean()),
            "rmse_std":float(scores["test_score"].std(ddof=1)),"rmse_train":float((-scores["train_score"]).mean())}
    resultados_derivadas.append(fila)
    with mlflow.start_run(run_name="features_"+nombre):
        mlflow.set_tags({"phase":"features","test_used":"false","model":"Ridge"})
        mlflow.log_param("derived_features",nombre)
        mlflow.log_metrics({k:v for k,v in fila.items() if k!='features'})
        mlflow.log_dict(split_manifest,"split_context.json")
resumen_derivadas = pd.DataFrame(resultados_derivadas)
resumen_derivadas["mejora_vs_sin_derivadas"] = resumen_derivadas.iloc[0].rmse_cv-resumen_derivadas.rmse_cv
salida = ROOT / "reports/features" / snapshot
salida.mkdir(parents=True,exist_ok=True)
resumen_derivadas.to_csv(salida / "ablation.csv",index=False)
display(resumen_derivadas.round(4))
print("Comparación exploratoria: los finalistas conservan los 173 predictores originales.")


2026/11/19 04:02:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run features_sin_derivadas at: http://mlflow:5000/#/experiments/1/runs/cb32a277fc964c11984fbb09dd5f47c2.


2026/11/19 04:02:53 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


2026/11/19 04:02:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run features_LOG1P_SELECTED_PAIR_COUNT at: http://mlflow:5000/#/experiments/1/runs/a4803be8f5e34c06af842b3dbdad9e68.


2026/11/19 04:02:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


2026/11/19 04:02:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run features_PARTNER_SIZE_BALANCE at: http://mlflow:5000/#/experiments/1/runs/9b15500327074db9a22f2e283d4c7f12.


2026/11/19 04:02:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


2026/11/19 04:02:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run features_LOG1P_SELECTED_PAIR_COUNT + PARTNER_SIZE_BALANCE at: http://mlflow:5000/#/experiments/1/runs/876d0b39e07c4bd7bd4b17a8bbf88011.


2026/11/19 04:02:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


,features,rmse_cv,rmse_std,rmse_train,mejora_vs_sin_derivadas
0,sin_derivadas,1.7980,0.1022,1.6456,0.0000
1,LOG1P_SELECTED_PAIR_COUNT,1.7985,0.1017,1.6455,-0.0005
2,PARTNER_SIZE_BALANCE,1.7878,0.0993,1.6365,0.0103
3,LOG1P_SELECTED_PAIR_COUNT + PARTNER_SIZE_BALANCE,1.7886,0.0983,1.6357,0.0094


Comparación exploratoria: los finalistas conservan los 173 predictores originales.
